## 🚀 Si en el futuro necesitas instalar más paquetes
# Siempre usa:/opt/homebrew/bin/python3 -m pip install nombre-del-paquete


In [1]:
# Siempre empezamos con un dataset para entrenar. Descargamos el dataset "tiny shakespeare"
# (texto plano con obras de Shakespeare, ideal para un modelo de lenguaje a nivel de carácter)
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt



--2026-05-14 15:09:53--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2026-05-14 15:09:54 (172 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# lo leemos para inspeccionarlo
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


In [3]:
print("length of dataset in characters: ", len(text)) # cantidad total de caracteres del dataset


length of dataset in characters:  1115394


In [4]:
# veamos los primeros 1000 caracteres para tener una idea del contenido
print(text[:1000])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# aquí están todos los caracteres únicos que aparecen en el texto: nuestro "vocabulario"
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# creamos un mapeo entre caracteres y números enteros (tokenización a nivel de carácter)
stoi = { ch:i for i,ch in enumerate(chars) } # string -> índice (encoder)
itos = { i:ch for i,ch in enumerate(chars) } # índice -> string (decoder)
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))


[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
import sys
print(sys.executable)



/usr/bin/python3


In [8]:
import sys
import torch

print("Python path:", sys.executable)
print("Torch version:", torch.__version__)


Python path: /usr/bin/python3
Torch version: 2.10.0+cpu


In [9]:
# ahora codificamos todo el dataset de texto y lo guardamos en un torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # los 1000 caracteres que vimos antes, ahora como los "ve" el GPT (números)


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [10]:
# separamos los datos en conjuntos de entrenamiento (train) y validación (val)
n = int(0.9*len(data)) # el primer 90% será para entrenar, el resto para validar
train_data = data[:n]
val_data = data[n:]


In [11]:
# 8 caracteres es un buen tamaño de "block_size" para empezar (longitud de contexto)
block_size = 8
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
# Ejemplo: dentro de un mismo bloque hay block_size ejemplos de entrenamiento "apilados",
# cada uno con un contexto distinto (de 1 hasta block_size caracteres) y su objetivo correspondiente.
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")


when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [14]:
torch.manual_seed(1337) # semilla para reproducibilidad
batch_size = 4 # cuántas secuencias independientes procesamos en paralelo
block_size = 8 # longitud máxima de contexto para las predicciones

def get_batch(split):
    # genera un pequeño batch de datos de entradas x y objetivos y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # posiciones iniciales aleatorias
    x = torch.stack([data[i:i+block_size] for i in ix]) # entradas
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # objetivos: la entrada desplazada un carácter
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [15]:
print(xb) # esta es la entrada que recibirá el transformer


tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [17]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# Modelo más simple posible: un modelo de lenguaje "bigram".
# Predice el siguiente carácter mirando únicamente el carácter actual (sin contexto adicional),
# usando una tabla de embeddings como tabla de consulta de logits.
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # cada token consulta directamente los logits del siguiente token en una tabla
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            # aplanamos batch y tiempo para que cross_entropy reciba (N, C) y (N,)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # genera texto de forma autoregresiva: predice, agrega, repite
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) # con vocab_size=65, la pérdida ideal de un modelo "adivinando al azar" sería ~ -ln(1/65) ≈ 4.17

# generamos texto con el modelo SIN entrenar: será básicamente ruido aleatorio
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [18]:
# creamos un optimizador de PyTorch (AdamW funciona muy bien para este tipo de modelos)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [24]:
batch_size = 32
for steps in range(100): # aumentar el número de pasos para mejores resultados...

    # tomamos un batch de datos
    xb, yb = get_batch('train')

    # evaluamos la pérdida
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) # limpiamos los gradientes del paso anterior
    loss.backward() # backpropagation: calcula los gradientes
    optimizer.step() # actualiza los parámetros del modelo

print(loss.item())


4.517106056213379


In [25]:
# generamos texto con el modelo bigram ya entrenado: sigue siendo bastante simple
# (solo mira el carácter anterior), pero ya no es ruido puro
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))



rJtttttRC !wlbGeZ;ATr&e?ONczPcBH&ooYTEmcUzMyeph s&
YCf
w:N
Y!xT:MjHQmIQnnhiAJa!maj C-
pdmGmZ!BL!vXaUIUgFPTA;ArzPLCSBk?KuQ!LJKyNR;AcXERCjsuDqgZYNNV&f3fI'zdW
rF;,IqHYZ;KCeDT'MpSZxap$pVKuFbfR?lv.?d,cPFKktkta'IV;I$N!B??hogCheCJZAJXmvvGX&o
VreiNSbuQ?BObQ?wUgOQCDo' -jgme:zyQqfp3HFM z;
p'P,Soiy,scY3fqxpHcizdZgUv.,Vq?wfv'wIRfa!
pSVjzXeuQhl?RT; Llgiyegq-fRykU'BHktDphSVbi.D.
VSA-wNyNzWW:DXx-bmZa!UPvWKATE?,S.ADWT CdfZxTX'd,S.P MlLb
pW?BGBzmV,
ohOE3tyORmRytiCBiq;zkzc;qG&v&AkqHU,Sq,iB?oVryxj?pa C&yOyS&fkNJMe


In [26]:
# ejemplo de juguete que ilustra cómo la multiplicación de matrices puede usarse
# para hacer una "agregación ponderada" (la base matemática de la self-attention)
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3)) # matriz triangular inferior de unos
a = a / torch.sum(a, 1, keepdim=True) # normalizamos cada fila para que sume 1 (promedio de los elementos anteriores)
b = torch.randint(0,10,(3,2)).float()
c = a @ b # a @ b = promedio acumulado de las filas de b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [27]:
# consideremos el siguiente ejemplo de juguete:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels (lote, tiempo, canales)
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [28]:
# queremos que x[b,t] = promedio_{i<=t} x[b,i]
# es decir, que cada posición "vea" el promedio de sí misma y de las anteriores (no del futuro)
# version 1: con bucles explícitos (lento, pero fácil de entender)
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)



In [29]:
# version 2: usando multiplicación de matrices para la agregación ponderada (mucho más rápido)
wei = torch.tril(torch.ones(T, T)) # máscara triangular inferior: cada fila "ve" solo posiciones <= t
wei = wei / wei.sum(1, keepdim=True) # normalizamos para que cada fila sea un promedio
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2) # verificamos que da el mismo resultado que la versión con bucles


True

In [30]:
# version 3: usando Softmax (equivalente a las anteriores, pero es la forma que usaremos en self-attention)
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # ponemos -inf donde no se permite "mirar" (futuro)
wei = F.softmax(wei, dim=-1) # softmax convierte esos -inf en ceros y normaliza el resto -> mismo promedio de antes
xbow3 = wei @ x
torch.allclose(xbow, xbow3) # de nuevo, el mismo resultado



True

In [31]:
# version 4: ¡self-attention!
# en vez de un promedio simple (pesos fijos), ahora los pesos "wei" se calculan dinámicamente
# a partir del contenido de cada token (queries y keys), permitiendo que cada token decida
# a cuáles tokens anteriores prestarle más atención.
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# veamos cómo una sola Head (cabeza) realiza self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)   # "qué información ofrezco"
query = nn.Linear(C, head_size, bias=False) # "qué información busco"
value = nn.Linear(C, head_size, bias=False) # "qué comparto si me prestan atención"
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # producto punto query·key -> afinidades (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # atención causal: no se puede mirar al futuro
wei = F.softmax(wei, dim=-1) # normaliza las afinidades a pesos que suman 1

v = value(x)
out = wei @ v # agregación ponderada de los "values" según los pesos de atención
#out = wei @ x

out.shape


torch.Size([4, 8, 16])

In [40]:
wei[0] # los pesos de atención del primer elemento del batch: cada fila muestra a qué presta atención cada token


tensor([[ 1.9857, -1.7370,  0.4189,  0.2985, -0.5451,  0.4942, -0.7267, -0.4810],
        [-0.6151,  0.7704,  0.1215,  0.1193, -1.0559, -0.1234,  0.3918, -0.2687],
        [ 0.4511,  0.6600,  0.8736, -0.5065,  0.8595,  0.2483,  0.7095, -0.3241],
        [ 1.4350, -0.5599,  1.2163, -0.0813,  1.7313,  0.3421, -0.3146, -0.9178],
        [-2.0204,  1.8716, -1.1214, -0.1317, -0.4320,  0.8461,  1.0991,  1.8651],
        [ 1.0000,  0.5394,  0.9807, -0.0900,  0.7364,  1.3018,  1.4779,  1.2385],
        [ 1.0542, -0.5249,  0.1258, -0.0781,  0.8236, -1.0546,  0.3601, -0.5679],
        [ 0.2587,  0.1620,  0.6471,  0.2837,  1.2641,  0.3890, -0.6218, -0.4601]])

Notes:

Attention is a communication mechanism. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
Each example across batch dimension is of course processed completely independently and never "talk" to each other
In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
"self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
"Scaled" attention additional divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [36]:
# ¿por qué escalamos por head_size**-0.5? para mantener la varianza de "wei" cercana a 1
# y evitar que el softmax se sature (se vuelva casi un one-hot) al inicio del entrenamiento
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5


In [37]:
k.var() # varianza de las keys (debería ser ~1)


tensor(1.0449)

In [38]:
q.var() # varianza de las queries (debería ser ~1)


tensor(1.0700)

In [39]:
wei.var() # gracias al escalado, la varianza de wei también queda cerca de 1


tensor(1.0918)

In [41]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1) # softmax "normal": distribución relativamente suave


tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [42]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot
# al multiplicar por un valor grande, el softmax se vuelve demasiado "afilado" y converge a un one-hot:
# esto es justamente lo que el escalado por sqrt(head_size) ayuda a evitar


tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [43]:
# Layer Normalization: normaliza cada muestra (fila) de forma independiente,
# en lugar de normalizar a través del batch como hace BatchNorm.
# Esto es lo que se usa dentro de los bloques transformer (ln1, ln2, ln_f).
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # media por fila (no por batch)
    xvar = x.var(1, keepdim=True) # varianza por fila
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normaliza a varianza unitaria
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch de 32 vectores de 100 dimensiones
x = module(x)
x.shape


torch.Size([32, 100])

In [44]:
x[:,0].mean(), x[:,0].std() # media y desvío de UNA característica a través de todo el batch
# (a diferencia de BatchNorm, LayerNorm no garantiza que esto dé exactamente 0 y 1,
# porque normaliza por fila, no por columna)


(tensor(0.1469), tensor(0.8803))

In [ ]:
# French yo english translation example

# les réseaux de neurones sont géniaux

In [46]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# ===================== Hiperparámetros =====================
batch_size = 64 # cuántas secuencias independientes procesamos en paralelo
block_size = 256 # longitud máxima de contexto para las predicciones
max_iters = 5000 # número total de pasos de entrenamiento
eval_interval = 500 # cada cuántos pasos medimos la pérdida en train/val
learning_rate = 3e-4 # tasa de aprendizaje del optimizador AdamW
device = 'cuda' if torch.cuda.is_available() else 'cpu' # usa GPU si está disponible
eval_iters = 200 # cuántos batches promediamos para estimar la pérdida
n_embd = 384 # dimensión de los embeddings (y del modelo en general)
n_head = 6 # número de cabezas de atención por bloque transformer
n_layer = 6 # número de bloques transformer apilados
dropout = 0.2 # probabilidad de dropout, ayuda a regularizar
# ------------

torch.manual_seed(1337) # semilla fija para resultados reproducibles

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# ===================== Tokenización a nivel de carácter =====================
# here are all the unique characters that occur in this text
chars = sorted(list(set(text))) # vocabulario: caracteres únicos del texto
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) } # string -> índice
itos = { i:ch for i,ch in enumerate(chars) } # índice -> string
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long) # todo el texto codificado como enteros
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # posiciones iniciales aleatorias
    x = torch.stack([data[i:i+block_size] for i in ix]) # entradas: bloques de "block_size" caracteres
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # objetivos: el mismo bloque desplazado un carácter
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad() # no necesitamos gradientes solo para evaluar
def estimate_loss():
    # promedia la pérdida sobre varios batches para una estimación menos ruidosa
    out = {}
    model.eval() # modo evaluación (desactiva dropout)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # vuelve a modo entrenamiento (reactiva dropout)
    return out

# ===================== Bloques del Transformer =====================
class Head(nn.Module):
    """ one head of self-attention """
    # Una sola "cabeza" de self-attention: decide cuánta atención prestarle
    # a los tokens anteriores (incluido el propio) para construir su representación.

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False) # "qué información ofrezco"
        self.query = nn.Linear(n_embd, head_size, bias=False) # "qué información busco"
        self.value = nn.Linear(n_embd, head_size, bias=False) # "qué comparto si me prestan atención"
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # máscara causal: no ver el futuro

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # producto punto query·key escalado -> (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # bloquea la atención hacia el futuro -> (B, T, T)
        wei = F.softmax(wei, dim=-1) # convierte los puntajes en pesos que suman 1 -> (B, T, T)
        wei = self.dropout(wei) # apaga aleatoriamente algunas conexiones de atención
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # combina los "values" según los pesos de atención -> (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    # Corre varias cabezas en paralelo y junta sus resultados; cada una puede
    # aprender a fijarse en relaciones distintas entre tokens.

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd) # vuelve a proyectar al tamaño original del embedding
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concatena las salidas de todas las cabezas
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    # Red feed-forward posicional: procesa cada token de forma independiente,
    # dándole al modelo capacidad de "pensar" sobre lo recolectado por la atención.

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # expande la dimensión (factor 4, como en el paper original)
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), # vuelve a comprimir a la dimensión original
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    # "Comunicación" = self-attention (los tokens intercambian información)
    # "Cómputo" = feed-forward (cada token procesa esa información por su cuenta)
    # Las conexiones residuales (x + ...) y las layer norms estabilizan el entrenamiento.

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head # cada cabeza recibe una porción de la dimensión total
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd) # normaliza antes de la atención (pre-norm)
        self.ln2 = nn.LayerNorm(n_embd) # normaliza antes del feed-forward (pre-norm)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # conexión residual: x + atención(norm(x))
        x = x + self.ffwd(self.ln2(x)) # conexión residual: x + feedforward(norm(x))
        return x

# super simple bigram model
# (el nombre quedó del modelo bigram original; en realidad ya es un modelo GPT completo
# con embeddings de posición y bloques transformer)
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # embedding por identidad del token
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # embedding por posición
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # bloques transformer apilados
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size) # proyecta al vocabulario (logits)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # embedding según qué carácter es -> (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # embedding según la posición -> (T,C)
        x = tok_emb + pos_emb # se suman: cada token "sabe" quién es y dónde está -> (B,T,C)
        x = self.blocks(x) # pasa por los bloques transformer -> (B,T,C)
        x = self.ln_f(x) # normalización final -> (B,T,C)
        logits = self.lm_head(x) # puntajes para cada carácter del vocabulario -> (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            # aplanamos batch y tiempo para que cross_entropy reciba (N, C) y (N,)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # genera texto carácter por carácter de forma autoregresiva: predice, agrega, repite
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:] # el modelo solo ve hasta block_size tokens de contexto
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # solo nos interesa la predicción del último paso -> (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # muestrea el siguiente carácter -> (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # lo agrega como contexto para el siguiente paso -> (B, T+1)
        return idx

# ===================== Entrenamiento =====================
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True) # limpia los gradientes del paso anterior
    loss.backward() # backpropagation: calcula los gradientes
    optimizer.step() # actualiza los parámetros del modelo

# ===================== Generación de texto con el modelo entrenado =====================
context = torch.zeros((1, 1), dtype=torch.long, device=device) # arranca desde un solo carácter "vacío"
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


10.788929 M parameters
step 0: train loss 4.2849, val loss 4.2823


KeyboardInterrupt: 